# Análise de Dados Públicos — Programa Luz para Todos

Este notebook executa a coleta, o tratamento e a análise exploratória dos dados oficiais do Ministério de Minas e Energia.

**Objetivos:** analisar a evolução temporal, comparar regiões/estados/municípios, avaliar a abrangência territorial e preparar dados para dashboards.

In [ ]:
# No Google Colab, descomente:
# !git clone https://github.com/gabrielavitorino/analise-luz-para-todos.git
# %cd analise-luz-para-todos
# !pip install -r requirements.txt

## 1. Coleta e processamento automatizados

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "run_pipeline.py"], check=True)

## 2. Leitura dos dados tratados

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

DATA = Path("data/processed")
dom = pd.read_csv(DATA / "domicilios_tratados.csv")
rec = pd.read_csv(DATA / "recursos_tratados.csv")

display(dom.head())
print(f"Linhas de domicílios: {len(dom):,}")
print(f"Estados atendidos: {dom['estado'].nunique()}")
print(f"Municípios atendidos: {dom['municipio'].nunique()}")
print(f"Período: {int(dom['ano'].min())} a {int(dom['ano'].max())}")

## 3. Qualidade dos dados

In [ ]:
qualidade = pd.DataFrame({
    "coluna": dom.columns,
    "tipo": dom.dtypes.astype(str).values,
    "nulos": dom.isna().sum().values,
    "percentual_nulos": (dom.isna().mean().values * 100).round(2),
    "valores_unicos": [dom[c].nunique(dropna=True) for c in dom.columns],
})
display(qualidade)

## 4. Indicadores gerais

In [ ]:
indicadores = {
    "Domicílios atendidos": dom["domicilios_atendidos"].sum(),
    "Estados atendidos": dom["estado"].nunique(),
    "Municípios atendidos": dom["municipio"].nunique(),
    "Ano inicial": int(dom["ano"].min()),
    "Ano final": int(dom["ano"].max()),
}
pd.Series(indicadores).to_frame("valor")

## 5. Evolução anual

In [ ]:
por_ano = dom.groupby("ano", as_index=False)["domicilios_atendidos"].sum()
fig = px.line(por_ano, x="ano", y="domicilios_atendidos", markers=True,
              title="Evolução anual de domicílios atendidos")
fig.show()

## 6. Distribuição por região

In [ ]:
por_regiao = (dom.groupby("regiao", as_index=False)["domicilios_atendidos"]
              .sum().sort_values("domicilios_atendidos"))
px.bar(por_regiao, x="domicilios_atendidos", y="regiao", orientation="h",
       title="Domicílios atendidos por região").show()

## 7. Ranking dos estados

In [ ]:
por_estado = (dom.groupby(["estado", "regiao"], as_index=False)["domicilios_atendidos"]
              .sum().sort_values("domicilios_atendidos", ascending=False))
px.bar(por_estado.head(15).sort_values("domicilios_atendidos"),
       x="domicilios_atendidos", y="estado", orientation="h",
       title="Top 15 estados por domicílios atendidos").show()
display(por_estado.head(15))

## 8. Abrangência nacional

In [ ]:
ufs_esperadas = {'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS',
                 'MG','PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO'}
ufs_encontradas = set(dom["estado"].dropna().unique())
print("UFs encontradas:", len(ufs_encontradas))
print("UFs sem registros:", sorted(ufs_esperadas - ufs_encontradas))
print("Abrange as 27 UFs?", ufs_esperadas.issubset(ufs_encontradas))

## 9. Municípios com mais atendimentos

In [ ]:
por_municipio = (dom.groupby(["municipio","estado","regiao"], as_index=False)
                 ["domicilios_atendidos"].sum()
                 .sort_values("domicilios_atendidos", ascending=False))
por_municipio["localidade"] = por_municipio["municipio"] + " - " + por_municipio["estado"]
top = por_municipio.head(20).sort_values("domicilios_atendidos")
px.bar(top, x="domicilios_atendidos", y="localidade", orientation="h",
       title="Top 20 municípios por domicílios atendidos").show()

## 10. Tipos de atendimento

In [ ]:
por_programa = (dom.groupby("programa", as_index=False)["domicilios_atendidos"]
                .sum().sort_values("domicilios_atendidos", ascending=False))
px.bar(por_programa, x="domicilios_atendidos", y="programa", orientation="h",
       title="Domicílios por tipo de atendimento").show()
display(por_programa)

## 11. Recursos aplicados e eficiência

In [ ]:
if "estado" in rec.columns:
    rec_estado = rec.groupby(["estado", "regiao"], as_index=False).agg(
        valor_pago_total=("valor_pago_total", "sum"),
        valor_contratado_total=("valor_contratado_total", "sum"),
    )
    combinado = por_estado.merge(rec_estado, on=["estado", "regiao"], how="left")
    combinado["custo_pago_por_domicilio"] = (
        combinado["valor_pago_total"] / combinado["domicilios_atendidos"]
    )
    px.scatter(
        combinado,
        x="domicilios_atendidos",
        y="valor_pago_total",
        size="valor_pago_total",
        hover_name="estado",
        color="regiao",
        title="Recursos pagos × domicílios atendidos por estado",
    ).show()
    display(combinado.sort_values("custo_pago_por_domicilio", ascending=False).head(10))
else:
    print("A base de recursos não possui a coluna estado.")

## 12. Conclusões

Após executar o notebook, substitua este texto pelos resultados calculados. Sugestões:

- Identificar o ano com maior quantidade de atendimentos.
- Apontar a região e o estado com maior participação.
- Confirmar se todas as unidades federativas estão presentes.
- Destacar os municípios com maior volume.
- Comparar recursos pagos e atendimentos.
- Registrar limitações dos dados e cuidados de interpretação.